# División de datos con Iris
**Taller de Fabiana Leonardo Rufino**  
¿Reconocerá el modelo una flor nueva? En parejas, vais a comparar dos modelos con validación y a medir una sola vez la opción elegida en prueba.

Ejecutad las celdas en orden. Deteneos antes de la sección 4 hasta que hayáis anotado vuestra decisión. El conjunto Iris viene incluido en scikit-learn.

## 1. Una fila, una flor
Cuatro medidas en centímetros forman **X**. La especie es la etiqueta **y**. Antes de ejecutar, pensad en algo que podríamos conocer al medir una flor nueva.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

iris = load_iris()
X, y = iris.data, iris.target
print(f"{len(y)} flores, {X.shape[1]} medidas por flor")
print("Primera fila (cm):", X[0], "Especie:", iris.target_names[y[0]])
print("Especies:", ", ".join(iris.target_names))

## 2. Dos repartos para obtener tres conjuntos
Reservad el 20 % de 150 flores para prueba. ¿Qué porcentaje de las 120 restantes hay que reservar para que validación también tenga 30? Repartimos los índices para poder comprobar que ninguna fila aparece dos veces.

In [ ]:
SEMILLA = 42
indices = np.arange(len(y))
desarrollo, test = train_test_split(
    indices, test_size=0.20, stratify=y, random_state=SEMILLA
)
train, validacion = train_test_split(
    desarrollo, test_size=0.25, stratify=y[desarrollo], random_state=SEMILLA
)

repartos = {"Train": train, "Validación": validacion, "Test": test}
for nombre, filas in repartos.items():
    recuentos = np.bincount(y[filas], minlength=3)
    print(f"{nombre:10} {len(filas):3} flores. Por especie: {recuentos.tolist()}")

filas_asignadas = np.concatenate([train, validacion, test])
assert len(filas_asignadas) == len(indices)
assert np.array_equal(np.sort(filas_asignadas), indices)

**En parejas.** Comprobad que la suma sea 150 y que los tres conjuntos contengan las tres especies. ¿Por qué importa repetir el reparto cuando comparamos modelos? Aquí usamos etiquetas para estratificar el reparto. Las predicciones de prueba se reservan para el final.

## 3. Comparación en validación
Mantendremos el reparto, el escalador y la regresión logística. Solo cambiaremos **C**, un parámetro que regula la intensidad de la regularización. Cuanto menor es C, más fuerte es la regularización. Antes de ejecutar, marcad la línea en la que el escalador aprende las medias.

In [ ]:
modelos = {}
aciertos_validacion = {}
for c in (0.01, 1.0):
    modelo = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=c, max_iter=1000)
    )
    modelo.fit(X[train], y[train])
    modelos[c] = modelo
    predicciones = modelo.predict(X[validacion])
    aciertos_validacion[c] = int(np.sum(predicciones == y[validacion]))
    print(f"C = {c:g}: {aciertos_validacion[c]} aciertos de {len(validacion)}")

**Pausa de decisión.** En una hoja, anotad la C que enviaríais al examen, el número de aciertos que apoya vuestra elección y qué conjunto sigue cerrado. El criterio fijado para esta práctica es elegir la mayor cifra de aciertos en validación. En caso de empate elegimos la C más pequeña.

In [ ]:
c_elegida = min(aciertos_validacion, key=lambda c: (-aciertos_validacion[c], c))
modelo_elegido = modelos[c_elegida]
print(f"Decisión registrada: C = {c_elegida:g}")

## 4. Apertura del test
Ejecutad esta celda cuando hayáis cerrado la decisión. Evaluamos el modelo ganador tal como se entrenó en la sección anterior. ¿Cuántos puntos porcentuales equivale a acertar una flor más en esta muestra?

In [ ]:
pred_final = modelo_elegido.predict(X[test])
aciertos = int(np.sum(pred_final == y[test]))
print(f"Prueba final: {aciertos} de {len(test)} ({aciertos / len(test):.1%})")
print(f"Cada flor equivale a {100 / len(test):.1f} puntos porcentuales.")

**Para debatir.** ¿Qué estiman esos 30 ejemplos? Si usáramos su nota para volver a escoger C, ¿de dónde saldría la siguiente evaluación independiente?

**Cierre oral.** Complete the sentence: “We kept the test set separate because…”

### Fuentes
- [Conjunto Iris](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html)
- [Validación y elección de modelos](https://scikit-learn.org/stable/modules/cross_validation.html)
- [Prevención de fugas de información](https://scikit-learn.org/stable/common_pitfalls.html)